# IOAI — 2025 Stage 3 Inpainting (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train/images'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-inpainting/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 인페인팅 — SIREN INR 모범답안

폴란드 AI 올림피아드 II · 2025 · 3단계. 영상 전체를 함수 `f(x,y,t)→(RGB,mask)` 로 표현(INR)해 잘린
사각형을 복원. **SIREN**(정현파 활성 MLP)로 59개 온전 프레임을 과적합 학습 → 10개 val 사각형을 예측.

**구조**: 입력 (x,y,t)∈[-1,1]³ → SIREN(sine 활성 MLP 5층·폭 256) → RGB(sigmoid) + mask(sigmoid).
영상이 시공간적으로 매끄러워 INR 이 미관측 프레임/영역을 보간한다.

**성능(val 10프레임, 실측)**: PSNR **26.3**(만점)·mask acc **0.97** → **≈98/100** (베이스라인 0점).

**제출**: `submission.npz` — `rgb(10,64,64,3)`·`mask(10,64,64)`.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, json, urllib.request, zipfile
if not os.path.exists("data/train/images"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-inpainting/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import cv2, numpy as np, torch, torch.nn as nn
dev = "cuda" if torch.cuda.is_available() else "cpu"
VIDEO_LENGTH = 79

def load_train():   # 59개 온전 프레임 -> 좌표(x,y,t), RGB, mask (픽셀 단위)
    C, RGB, M = [], [], []
    for fn in sorted(os.listdir("data/train/images")):
        img = cv2.cvtColor(cv2.imread(f"data/train/images/{fn}"), cv2.COLOR_BGR2RGB).astype("float32")/255.0
        msk = cv2.imread(f"data/train/masks/{fn.replace('.jpg','.png')}", 0).astype("float32")/255.0
        h, w = img.shape[:2]
        gy, gx = np.meshgrid(np.linspace(-1,1,h), np.linspace(-1,1,w), indexing="ij")
        t = int(os.path.splitext(fn)[0]) * 2.0 / (VIDEO_LENGTH - 1)
        C.append(np.stack([gx.ravel(), gy.ravel(), np.full(h*w, t)], -1))
        RGB.append(img.reshape(-1,3)); M.append(msk.reshape(-1,1))
    return (torch.tensor(np.concatenate(C), dtype=torch.float32),
            torch.tensor(np.concatenate(RGB), dtype=torch.float32),
            torch.tensor(np.concatenate(M), dtype=torch.float32))

def val_rect_coords():  # 10개 val 프레임의 사각형 격자 좌표 (예측 대상; 정답 미공개)
    r = json.load(open("data/val_rectangles.json")); frames = sorted(r); out = []
    for fn in frames:
        c = r[fn]; x1,y1,x2,y2 = c["x1"],c["y1"],c["x2"],c["y2"]; w,h = x2-x1, y2-y1
        gy, gx = np.meshgrid(np.linspace(y1/256*2-1, y2/256*2-1, h),
                             np.linspace(x1/256*2-1, x2/256*2-1, w), indexing="ij")
        t = int(os.path.splitext(fn)[0]) * 2.0 / (VIDEO_LENGTH - 1)
        out.append((fn, torch.tensor(np.stack([gx.ravel(), gy.ravel(), np.full(h*w, t)], -1), dtype=torch.float32)))
    return frames, out

frames, val_coords = val_rect_coords()
print("val frames:", len(frames))


In [ ]:
# SIREN (정현파 활성 INR)
class Sine(nn.Module):
    def __init__(s, w0=30.0): super().__init__(); s.w0=w0
    def forward(s, x): return torch.sin(s.w0 * x)

class SirenINR(nn.Module):
    def __init__(s, din=3, dh=256, nl=5):
        super().__init__(); layers=[]; d=din
        for i in range(nl):
            lin = nn.Linear(d, dh)
            with torch.no_grad():
                if i==0: lin.weight.uniform_(-1/d, 1/d)
                else:    lin.weight.uniform_(-np.sqrt(6/d)/30, np.sqrt(6/d)/30)
            layers += [lin, Sine(30.0)]; d = dh
        s.body = nn.Sequential(*layers); s.rgb = nn.Linear(d, 3); s.msk = nn.Linear(d, 1)
    def forward(s, x):
        h = s.body(x); return torch.sigmoid(s.rgb(h)), torch.sigmoid(s.msk(h))

def train_siren(iters=6000, bs=65536, lr=1e-4):
    C, RGB, M = load_train(); C, RGB, M = C.to(dev), RGB.to(dev), M.to(dev)
    print("train points:", C.shape[0])
    net = SirenINR().to(dev); opt = torch.optim.Adam(net.parameters(), lr=lr); bce = nn.BCELoss()
    for it in range(iters):
        idx = torch.randint(0, C.shape[0], (bs,), device=dev)
        rp, mp = net(C[idx])
        loss = nn.functional.mse_loss(rp, RGB[idx]) + 0.5 * bce(mp, M[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 2000 == 1999: print(f"it {it+1} loss {loss.item():.4f}", flush=True)
    return net.eval()

model = train_siren()


In [ ]:
# 각 val 사각형 예측 -> submission.npz (rgb, mask; 프레임 정렬순, 각 64x64)
model.eval(); rgbs=[]; masks=[]
with torch.no_grad():
    for fn, coords in val_coords:
        rp, mp = model(coords.to(dev))
        rgbs.append(rp.cpu().numpy().reshape(64,64,3))
        masks.append(mp.cpu().numpy().reshape(64,64))
np.savez_compressed("submission.npz",
    rgb=np.stack(rgbs).astype("float32"), mask=np.stack(masks).astype("float32"))
print("submission.npz 저장:", len(rgbs), "프레임")


### 정리
- SIREN INR 로 영상을 f(x,y,t) 로 표현 → 잘린 사각형을 시공간 보간으로 복원. PSNR 26·acc 0.97 → ≈98점.
- **핵심**: 정현파 활성이 고주파 디테일을 잡아 INR 이 이미지를 선명히 표현(ReLU 대비 우수).


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)